# Red bayesiana: falla de una computadora

Ejemplo para calcular la probabilidad de que una computadora falle considerando el uso, la ventilación y la temperatura.

```text
UsoIntensivo -----\
                  > Temperatura ---> FallaSistema
Ventilacion ------/
```


In [ ]:
# Instalar pgmpy si todavía no está instalado
#!pip install pgmpy


In [ ]:
# Librerías para crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination


In [ ]:
# Crear la estructura de la red
model = DiscreteBayesianNetwork([
    ("UsoIntensivo", "Temperatura"),
    ("Ventilacion", "Temperatura"),
    ("Temperatura", "FallaSistema")
])

print(model.edges())


In [ ]:
# Probabilidad de uso intensivo
# 35% de uso intensivo y 65% de uso normal
cpd_uso = TabularCPD(
    variable="UsoIntensivo",
    variable_card=2,
    values=[[0.65], [0.35]],
    state_names={"UsoIntensivo": ["No", "Sí"]}
)


In [ ]:
# Probabilidad del estado de la ventilación
# 80% buena y 20% deficiente
cpd_ventilacion = TabularCPD(
    variable="Ventilacion",
    variable_card=2,
    values=[[0.80], [0.20]],
    state_names={"Ventilacion": ["Buena", "Deficiente"]}
)


In [ ]:
# Probabilidad de la temperatura según el uso y la ventilación
cpd_temperatura = TabularCPD(
    variable="Temperatura",
    variable_card=2,
    values=[
        [0.95, 0.70, 0.55, 0.15],  # Normal
        [0.05, 0.30, 0.45, 0.85]   # Alta
    ],
    evidence=["UsoIntensivo", "Ventilacion"],
    evidence_card=[2, 2],
    state_names={
        "Temperatura": ["Normal", "Alta"],
        "UsoIntensivo": ["No", "Sí"],
        "Ventilacion": ["Buena", "Deficiente"]
    }
)


In [ ]:
# Probabilidad de falla según la temperatura
cpd_falla = TabularCPD(
    variable="FallaSistema",
    variable_card=2,
    values=[
        [0.98, 0.30],  # No falla
        [0.02, 0.70]   # Sí falla
    ],
    evidence=["Temperatura"],
    evidence_card=[2],
    state_names={
        "FallaSistema": ["No", "Sí"],
        "Temperatura": ["Normal", "Alta"]
    }
)


In [ ]:
# Agregar las probabilidades al modelo y verificar
model.add_cpds(cpd_uso, cpd_ventilacion, cpd_temperatura, cpd_falla)

print("Modelo válido:", model.check_model())
print("Relaciones:", list(model.edges()))


In [ ]:
# Crear el proceso de inferencia
inferencia = VariableElimination(model)

# Probabilidad general de que ocurra una falla
resultado_general = inferencia.query(
    variables=["FallaSistema"]
)

print("Probabilidad general de falla:")
print(resultado_general)


In [ ]:
# Probabilidad de falla si hay uso intensivo y ventilación deficiente
resultado_falla = inferencia.query(
    variables=["FallaSistema"],
    evidence={
        "UsoIntensivo": "Sí",
        "Ventilacion": "Deficiente"
    }
)

print("Probabilidad de falla con uso intensivo y ventilación deficiente:")
print(resultado_falla)


In [ ]:
# Aplicando inferencia: si sabemos que ocurrió una falla,
# calculamos la probabilidad de que haya existido uso intensivo
resultado_bayes = inferencia.query(
    variables=["UsoIntensivo"],
    evidence={"FallaSistema": "Sí"}
)

print("Probabilidad de uso intensivo sabiendo que hubo una falla:")
print(resultado_bayes)


## Conclusión

La red bayesiana permite calcular la probabilidad de una falla y actualizar el resultado cuando se conoce nueva evidencia.

En este ejemplo, el uso intensivo y una ventilación deficiente aumentan la probabilidad de que la computadora falle.
